# ROGII: One Coefficient, Three RMSE Optima

Pooled row-wise RMSE, macro well RMSE, and p95 well RMSE answer different questions. Optimizing the wrong aggregation can silently select the wrong global constant even when every prediction and split is otherwise leakage-safe.

This notebook uses a deliberately small model so the metric mechanics stay visible. Starting from the last-visible-TVT anchor, we fit a clipped direct-TVT slope on the final 30 visible rows and multiply it by one global shrinkage coefficient `alpha`. We then evaluate all 773 wells at their exact supplied missing boundary.

The notebook derives the pooled optimum analytically, finds the macro and p95 optima on a dense grid, performs five-fold out-of-well fitting, and reports tail trade-offs. It creates no submission and uses no leaderboard feedback.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data was not found')
TRAIN_DIR = DATA_ROOT / 'train'
WINDOW = 30
SLOPE_LIMIT = 0.10
FOLDS = 5
print('Data root:', DATA_ROOT)
print('Training wells:', len(list(TRAIN_DIR.glob('*__horizontal_well.csv'))))

## True-start protocol and sufficient statistics

For one suffix row, the candidate is

`prediction(alpha) = anchor + alpha * clipped_prefix_slope * MD_distance`.

If `e` is anchor error and `d` is the raw trend adjustment, each well's SSE is exactly

`SSE_i(alpha) = c_i + 2 * alpha * b_i + alpha^2 * a_i`,

where `a_i = d dot d`, `b_i = e dot d`, and `c_i = e dot e`. Storing these four well-level statistics plus row count is sufficient to evaluate any global alpha without retaining millions of row predictions.

In [ ]:
def boundary(values):
    observed = values.notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] < 2 or observed[missing[0]:].any():
        raise ValueError('Expected a visible TVT prefix and one contiguous missing suffix')
    return int(missing[0])

def tail_slope(md, tvt_prefix, window=WINDOW):
    x = md[-window:]
    y = tvt_prefix[-window:]
    centered = x - x.mean()
    denominator = float(centered @ centered)
    slope = 0.0 if denominator <= 1e-12 else float(centered @ (y - y.mean()) / denominator)
    return float(np.clip(slope, -SLOPE_LIMIT, SLOPE_LIMIT))

rows = []
for path in sorted(TRAIN_DIR.glob('*__horizontal_well.csv')):
    frame = pd.read_csv(path, usecols=['MD', 'TVT', 'TVT_input'])
    cut = boundary(frame['TVT_input'])
    md = frame['MD'].to_numpy(dtype=float)
    truth = frame['TVT'].to_numpy(dtype=float)
    visible = frame['TVT_input'].to_numpy(dtype=float)[:cut]
    anchor = float(visible[-1])
    anchor_error = anchor - truth[cut:]
    slope = tail_slope(md[:cut], visible)
    adjustment = slope * (md[cut:] - md[cut - 1])
    rows.append({
        'well_id': path.name.removesuffix('__horizontal_well.csv'),
        'n_eval': len(anchor_error),
        'slope': slope,
        'a': float(adjustment @ adjustment),
        'b': float(anchor_error @ adjustment),
        'c': float(anchor_error @ anchor_error),
    })

stats = pd.DataFrame(rows)
stats['fold'] = np.arange(len(stats)) % FOLDS
print('Wells:', len(stats))
print('Evaluation rows:', int(stats['n_eval'].sum()))
display(stats.head())

In [ ]:
def well_sse(alpha, frame=stats):
    return frame['c'] + 2.0 * alpha * frame['b'] + alpha * alpha * frame['a']

def metrics(alpha, frame=stats):
    sse = well_sse(alpha, frame).clip(lower=0.0)
    per_well = np.sqrt(sse / frame['n_eval'])
    return {
        'alpha': float(alpha),
        'pooled_rmse': math.sqrt(float(sse.sum()) / int(frame['n_eval'].sum())),
        'macro_rmse': float(per_well.mean()),
        'median_well': float(per_well.median()),
        'p95_well': float(per_well.quantile(0.95)),
        'worst_well': float(per_well.max()),
    }

denominator = float(stats['a'].sum())
pooled_alpha = float(np.clip(-stats['b'].sum() / denominator, 0.0, 1.0))
grid = np.linspace(0.0, 0.10, 501)
grid_metrics = pd.DataFrame([metrics(alpha) for alpha in grid])
macro_alpha = float(grid_metrics.loc[grid_metrics['macro_rmse'].idxmin(), 'alpha'])
p95_alpha = float(grid_metrics.loc[grid_metrics['p95_well'].idxmin(), 'alpha'])
print('Analytic pooled optimum:', pooled_alpha)
print('Grid macro optimum:', macro_alpha)
print('Grid p95 optimum:', p95_alpha)

## The three optima are close, but not interchangeable

A small numerical difference is still a different decision rule. The official competition score weights wells in proportion to suffix rows, so the pooled optimum is the relevant global coefficient. Macro RMSE gives every well equal weight. p95 focuses on the upper tail and is noisier because its identity can change as curves cross.

In [ ]:
choices = [
    ('anchor alpha=0', 0.0),
    ('pooled optimum', pooled_alpha),
    ('macro optimum', macro_alpha),
    ('p95 optimum', p95_alpha),
]
comparison = pd.DataFrame([{'selection_rule': name, **metrics(alpha)} for name, alpha in choices])
display(comparison.set_index('selection_rule').style.format({
    'alpha': '{:.5f}',
    'pooled_rmse': '{:.5f}',
    'macro_rmse': '{:.5f}',
    'median_well': '{:.5f}',
    'p95_well': '{:.5f}',
    'worst_well': '{:.5f}',
}))

## Fit the official objective out of well

The analytic coefficient is a fitted parameter even though it has only one dimension. We therefore learn it on four well folds and apply it only to the held-out fold. This checks parameter stability without random row leakage.

In [ ]:
fold_rows = []
oof_sse = 0.0
for fold in range(FOLDS):
    train = stats['fold'] != fold
    valid = ~train
    fold_alpha = float(np.clip(
        -stats.loc[train, 'b'].sum() / stats.loc[train, 'a'].sum(),
        0.0, 1.0,
    ))
    validation = stats.loc[valid]
    validation_sse = well_sse(fold_alpha, validation)
    oof_sse += float(validation_sse.sum())
    fold_rows.append({
        'fold': fold,
        'alpha': fold_alpha,
        'validation_wells': int(valid.sum()),
        'validation_pooled_rmse': math.sqrt(float(validation_sse.sum()) / int(validation['n_eval'].sum())),
    })
fold_table = pd.DataFrame(fold_rows)
display(fold_table.style.format({'alpha': '{:.5f}', 'validation_pooled_rmse': '{:.4f}'}))
print('OOF pooled RMSE:', math.sqrt(oof_sse / int(stats['n_eval'].sum())))
print('Full-fit pooled RMSE:', metrics(pooled_alpha)['pooled_rmse'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
objectives = [
    ('pooled_rmse', 'Pooled row-wise RMSE', pooled_alpha),
    ('macro_rmse', 'Macro well RMSE', macro_alpha),
    ('p95_well', 'p95 well RMSE', p95_alpha),
]
for ax, (column, title, optimum) in zip(axes, objectives):
    ax.plot(grid_metrics['alpha'], grid_metrics[column], color='#2a6fbb')
    ax.axvline(optimum, color='#c23b22', linestyle='--', label=f'optimum={optimum:.4f}')
    ax.set(title=title, xlabel='trend shrinkage alpha', ylabel='RMSE')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
audit = stats[['well_id', 'n_eval', 'slope']].copy()
for name, alpha in [('anchor', 0.0), ('pooled', pooled_alpha), ('p95', p95_alpha)]:
    audit[f'rmse_{name}'] = np.sqrt(well_sse(alpha) / stats['n_eval'])
audit['pooled_minus_anchor'] = audit['rmse_pooled'] - audit['rmse_anchor']
print('Wells helped by pooled-optimal trend:', int((audit['pooled_minus_anchor'] < 0).sum()), '/', len(audit))
display(audit.nlargest(10, 'pooled_minus_anchor'))
display(audit.nsmallest(10, 'pooled_minus_anchor'))

## Practical lessons

- Match model selection to the official pooled row-wise objective; macro and p95 are diagnostics, not substitutes.
- Still report macro and tail metrics. A pooled improvement can move p95 in the wrong direction.
- Fit even one global constant out of well. Low dimensionality does not make target-derived tuning free.
- Do not spend leaderboard submissions resolving nearby coefficients when the local objective is analytic and the OOF folds are stable.
- Keep the anchor. This tiny trend is a useful metric laboratory, but its gain is much smaller than the remaining geological branch-error problem.